# Creación de un Random Forest

En esta sección vamos a crear un modelo Random Forest (bosque aleatorio) para resolver un problema de clasificación/regresión. Incluye los siguientes pasos:

- Cargar y limpiar los datos.
- Seleccionar características y dividir en entrenamiento/prueba.
- Entrenar el modelo Random Forest.
- Evaluar rendimiento (matriz de confusión, precisión, RMSE, etc.).
- Ajustar hiperparámetros (GridSearch/RandomizedSearch).
- Interpretar importancia de variables y guardar el modelo para uso posterior.
---

# Paso 1: Cargamso y dividimos DataSet

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
from skopt import BayesSearchCV


In [14]:
df = pd.read_csv(r'..\..\datasets\pima_indian_diabetes_dataset\cleaned_dataset.csv')

X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Paso 2: Optimización de hiperparámetros

En este paso vamos a buscar la mejor configuración de hiperparámetros para los modelos (Random Forest y Árbol de Decisión) usando búsqueda bayesiana. Objetivos y pasos principales:

- Objetivo: maximizar la métrica ROC AUC en validación cruzada para obtener modelos más robustos.
- Método: BayesSearchCV sobre espacios de búsqueda definidos para cada modelo (n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, bootstrap, etc.).
- Configuración típica: cv=5, scoring='roc_auc', n_iter (p. ej. 20) y random_state para reproducibilidad. Usar n_jobs=-1 si se desea paralelizar.
- Flujo:
    1. Definir los search_spaces para RF y DT.
    2. Instanciar BayesSearchCV con el estimador correspondiente.
    3. Ajustar (fit) con X_train, y_train.
    4. Extraer best_params_ y crear los modelos finales con esos parámetros.
    5. Evaluar en X_test (accuracy, precision, recall, F1, ROC AUC, matriz de confusión) y revisar importancia de variables.
- Consideraciones: controlar n_iter y tiempo de cómputo, verificar sobreajuste comparando train/val, guardar modelos y resultados (joblib/pickle) y revisar la convergencia de la búsqueda.

In [16]:
#creamos el modelo de Random Forest
rf_classifier = RandomForestClassifier(random_state=42)
#Optimización de hiperparámetros con BayesSearchCV
search_space = {
    'n_estimators': (10, 500),
    'max_depth': (3, 50),  # Replaced None with 50 as an upper bound
    'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 8),
    'max_features': ['sqrt', 'log2'],  # Removed 'auto' as it is not valid
    'bootstrap': [True, False]
}
bayes_search = BayesSearchCV(estimator=rf_classifier, search_spaces=search_space, cv=5, scoring='roc_auc', verbose=3, n_iter=20, random_state=42)

# Es necesario ajustar (fit) el BayesSearchCV antes de acceder a best_params_
bayes_search.fit(X_train, y_train)
best_params = bayes_search.best_params_

#creamos el modelo de Arbol de Decision
dt_classifier = DecisionTreeClassifier(random_state=42)
#optimización de hiperparámetros con BayesSearchCV
dt_search_space = {
    'max_depth': (3, 50),  # Replaced None with 50 as an upper bound
    'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 8),
    'max_features': ['sqrt', 'log2']  # Removed 'auto' as it is not valid
}
dt_bayes_search = BayesSearchCV(estimator=dt_classifier, search_spaces=dt_search_space, cv=5, scoring='roc_auc', verbose=3, n_iter=20, random_state=42)

# Ajustar el BayesSearchCV del árbol antes de obtener los mejores parámetros
dt_bayes_search.fit(X_train, y_train)
dt_best_params = dt_bayes_search.best_params_

print("Mejores hiperparámetros para Random Forest:", best_params)
print("Mejores hiperparámetros para Árbol de Decisión:", dt_best_params)


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END bootstrap=True, max_depth=37, max_features=log2, min_samples_leaf=3, min_samples_split=14, n_estimators=213;, score=0.881 total time=   0.7s
[CV 2/5] END bootstrap=True, max_depth=37, max_features=log2, min_samples_leaf=3, min_samples_split=14, n_estimators=213;, score=0.812 total time=   0.8s
[CV 3/5] END bootstrap=True, max_depth=37, max_features=log2, min_samples_leaf=3, min_samples_split=14, n_estimators=213;, score=0.807 total time=   0.7s
[CV 4/5] END bootstrap=True, max_depth=37, max_features=log2, min_samples_leaf=3, min_samples_split=14, n_estimators=213;, score=0.868 total time=   0.6s
[CV 5/5] END bootstrap=True, max_depth=37, max_features=log2, min_samples_leaf=3, min_samples_split=14, n_estimators=213;, score=0.892 total time=   0.8s
Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END bootstrap=False, max_depth=45, max_features=sqrt, min_samples_leaf=8, min_samples_split=18, n_estim

### Paso 3: Entrenamiento y prueba con los mejores hiperparámetros (validación cruzada k=5)

- Objetivo: entrenar los modelos con best_params encontrados y evaluar su rendimiento robusto usando validación cruzada de 5 pliegues (cv=5).
- Flujo:
    1. Instanciar los modelos finales con los mejores parámetros (p. ej. `rf_best = RandomForestClassifier(**best_params, random_state=42)` y `dt_best = DecisionTreeClassifier(**dt_best_params, random_state=42)`).
    2. Realizar validación cruzada en el conjunto de entrenamiento (X_train, y_train) con cv=5 — usar `cross_validate` para obtener medias y desviaciones (scoring: 'roc_auc', 'accuracy', 'precision', 'recall', 'f1') para comparar desempeño entre pliegues.
    3. Ajustar (fit) cada modelo en todo X_train y predecir sobre X_test.
    4. Evaluar en el conjunto de prueba: accuracy, precision, recall, F1, ROC AUC (usar probabilidades para ROC), y matriz de confusión. Graficar curva ROC y matriz de confusión si es útil.
    5. Comparar métricas de validación cruzada (mean ± std) con las métricas en test para detectar sobreajuste o subajuste.
    6. Guardar modelos entrenados y resultados (ej. `joblib.dump`) y documentar las importancias de características (Random Forest).
- Consideraciones prácticas: usar `n_jobs=-1` cuando sea posible, mantener `random_state` para reproducibilidad y considerar calibración de probabilidades si se necesita una mejor estimación de scores.

In [17]:
# Crear modelos con los mejores hiperparámetros encontrados y entrenarlos
rf_best = RandomForestClassifier(**best_params, random_state=42)
dt_best = DecisionTreeClassifier(**dt_best_params, random_state=42)


In [18]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Configuración de métricas y CV
scoring = ['roc_auc', 'accuracy', 'precision', 'recall', 'f1']
cv = 5

# Validación cruzada en el conjunto de entrenamiento
rf_cv = cross_validate(rf_best, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
dt_cv = cross_validate(dt_best, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)

# Función auxiliar para mostrar resultados mean ± std
def print_cv_results(name, cv_results):
    print(f"\n{name} - Cross-validation (cv={cv}):")
    metrics = ['roc_auc', 'accuracy', 'precision', 'recall', 'f1']
    for m in metrics:
        test_key = f'test_{m}'
        train_key = f'train_{m}'
        test_mean = cv_results[test_key].mean()
        test_std = cv_results[test_key].std()
        train_mean = cv_results[train_key].mean()
        train_std = cv_results[train_key].std()
        print(f" {m}: test {test_mean:.4f} ± {test_std:.4f} | train {train_mean:.4f} ± {train_std:.4f}")

print_cv_results("RandomForest (cv)", rf_cv)
print_cv_results("DecisionTree (cv)", dt_cv)

# Entrenar en todo X_train y evaluar en X_test
rf_best.fit(X_train, y_train)
dt_best.fit(X_train, y_train)


def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    # Algunas implementaciones no dan predict_proba (aunque ambos modelos sí lo hacen), por seguridad:
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        roc = roc_auc_score(y_test, y_proba)
    except Exception:
        roc = float('nan')
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n{name} - Test set results:")
    print(f" Accuracy: {acc:.4f}")
    print(f" Precision: {prec:.4f}")
    print(f" Recall: {rec:.4f}")
    print(f" F1: {f1:.4f}")
    print(f" ROC AUC: {roc:.4f}")
    print(" Confusion matrix:")
    print(cm)

evaluate_model("RandomForest (final)", rf_best, X_test, y_test)
evaluate_model("DecisionTree (final)", dt_best, X_test, y_test)


RandomForest (cv) - Cross-validation (cv=5):
 roc_auc: test 0.8604 ± 0.0381 | train 0.9164 ± 0.0064
 accuracy: test 0.7925 ± 0.0405 | train 0.8419 ± 0.0081
 precision: test 0.7250 ± 0.0730 | train 0.8187 ± 0.0073
 recall: test 0.5268 ± 0.1200 | train 0.6289 ± 0.0265
 f1: test 0.6050 ± 0.1038 | train 0.7111 ± 0.0186

DecisionTree (cv) - Cross-validation (cv=5):
 roc_auc: test 0.7926 ± 0.0621 | train 0.8538 ± 0.0161
 accuracy: test 0.7700 ± 0.0498 | train 0.8043 ± 0.0201
 precision: test 0.7136 ± 0.1105 | train 0.8096 ± 0.0667
 recall: test 0.4426 ± 0.1571 | train 0.5054 ± 0.1485
 f1: test 0.5293 ± 0.1412 | train 0.6031 ± 0.0879

RandomForest (final) - Test set results:
 Accuracy: 0.7468
 Precision: 0.8095
 Recall: 0.5152
 F1: 0.6296
 ROC AUC: 0.8597
 Confusion matrix:
[[42  4]
 [16 17]]

DecisionTree (final) - Test set results:
 Accuracy: 0.6835
 Precision: 0.9000
 Recall: 0.2727
 F1: 0.4186
 ROC AUC: 0.7197
 Confusion matrix:
[[45  1]
 [24  9]]
